In [1]:
import numpy as np


In [2]:
import pandas as pd
train = pd.read_csv(
    "data/train.csv",
    parse_dates=["Date"],
    dtype={"StateHoliday": str},
    low_memory=False
)

store = pd.read_csv("data/store.csv")

data = train.merge(store, on="Store", how="left")

cutoff_date = "2015-07-01"

model_train = data[data["Date"] < cutoff_date].copy()
model_test = data[data["Date"] >= cutoff_date].copy()

y_test_true = model_test["Sales"].copy()

model_test_without_sales = model_test.drop(columns=["Sales"])

In [3]:
print(data["Date"].min())
print(data["Date"].max())

2013-01-01 00:00:00
2015-07-31 00:00:00


In [4]:
cutoff_date = "2015-07-01"

model_train = data[data["Date"] < cutoff_date].copy()
model_test = data[data["Date"] >= cutoff_date].copy()

y_test_true = model_test["Sales"].copy()
model_test_without_sales = model_test.drop(columns=["Sales"])

print("Training period:")
print(model_train["Date"].min(), model_train["Date"].max())

print("Hidden test period:")
print(model_test["Date"].min(), model_test["Date"].max())

print(model_train.shape)
print(model_test_without_sales.shape)

Training period:
2013-01-01 00:00:00 2015-06-30 00:00:00
Hidden test period:
2015-07-01 00:00:00 2015-07-31 00:00:00
(982644, 18)
(34565, 17)


In [5]:
print(model_train["Sales"].head())
print(y_test_true.head())
print(model_test_without_sales.columns)


34565     5735
34566     9863
34567    13261
34568    13106
34569     6635
Name: Sales, dtype: int64
0     5263
1     6064
2     8314
3    13995
4     4822
Name: Sales, dtype: int64
Index(['Store', 'DayOfWeek', 'Date', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval'],
      dtype='str')


In [6]:
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    return np.sqrt(
        np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2)
    )

In [7]:
store_average_sales = (
    model_train[model_train["Open"] == 1]
    .groupby("Store")["Sales"]
    .mean()
)

global_average_sales = model_train[model_train["Open"] == 1]["Sales"].mean()

baseline_pred = model_test_without_sales["Store"].map(store_average_sales)
baseline_pred = baseline_pred.fillna(global_average_sales)

baseline_pred = np.where(
    model_test_without_sales["Open"] == 0,
    0,
    baseline_pred
)

baseline_score = rmspe(y_test_true, baseline_pred)

print("Baseline RMSPE:", baseline_score)

Baseline RMSPE: 0.3071347823551685


In [8]:
print("Baseline:", baseline_pred)

Baseline: [ 4768.68435013  4953.88771466  6947.77393617 ...  6625.00792602
 20575.95376486  6276.71352785]


In [9]:
def add_date_features(df):
    df = df.copy()

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)
    df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)

    return df

model_train_fe = add_date_features(model_train)
model_test_fe = add_date_features(model_test_without_sales)

In [10]:
def clean_missing_values(df):
    df = df.copy()

    df["CompetitionDistance"] = df["CompetitionDistance"].fillna(999999)

    for col in [
        "CompetitionOpenSinceMonth",
        "CompetitionOpenSinceYear",
        "Promo2SinceWeek",
        "Promo2SinceYear",
        "PromoInterval",
    ]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df["Open"] = df["Open"].fillna(1)

    return df

model_train_fe = clean_missing_values(model_train_fe)
model_test_fe = clean_missing_values(model_test_fe)

In [11]:
def add_competition_features(df):
    df = df.copy()

    df["CompetitionOpen"] = 12 * (
        df["Year"] - df["CompetitionOpenSinceYear"]
    ) + (
        df["Month"] - df["CompetitionOpenSinceMonth"]
    )

    df.loc[
        (df["CompetitionOpenSinceYear"] == 0)
        | (df["CompetitionOpenSinceMonth"] == 0),
        "CompetitionOpen"
    ] = 0

    df["CompetitionOpen"] = df["CompetitionOpen"].clip(lower=0)

    return df

model_train_fe = add_competition_features(model_train_fe)
model_test_fe = add_competition_features(model_test_fe)

In [12]:
def add_promo2_features(df):
    df = df.copy()

    df["Promo2Open"] = 12 * (
        df["Year"] - df["Promo2SinceYear"]
    ) + (
        df["WeekOfYear"] - df["Promo2SinceWeek"]
    ) / 4.0

    df.loc[
        (df["Promo2"] == 0)
        | (df["Promo2SinceYear"] == 0)
        | (df["Promo2SinceWeek"] == 0),
        "Promo2Open"
    ] = 0

    df["Promo2Open"] = df["Promo2Open"].clip(lower=0)

    return df

model_train_fe = add_promo2_features(model_train_fe)
model_test_fe = add_promo2_features(model_test_fe)

In [13]:
categorical_cols = [
    "StoreType",
    "Assortment",
    "StateHoliday",
    "PromoInterval"
]

combined = pd.concat(
    [
        model_train_fe.drop(columns=["Sales", "Customers"], errors="ignore"),
        model_test_fe.drop(columns=["Customers"], errors="ignore")
    ],
    axis=0
)

combined_encoded = pd.get_dummies(
    combined,
    columns=categorical_cols,
    dummy_na=True
)

X_train_all = combined_encoded.iloc[:len(model_train_fe)].copy()
X_test_all = combined_encoded.iloc[len(model_train_fe):].copy()

y_train_all = model_train_fe["Sales"].copy()

In [14]:
drop_cols = ["Date"]

X_train_all = X_train_all.drop(columns=drop_cols, errors="ignore")
X_test_all = X_test_all.drop(columns=drop_cols, errors="ignore")

In [15]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from xgboost import XGBRegressor

open_train_mask = (model_train_fe["Open"] == 1) & (model_train_fe["Sales"] > 0)

X_train_open = X_train_all.loc[open_train_mask].copy()
y_train_open = np.log1p(y_train_all.loc[open_train_mask])

model = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_open, y_train_open)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [17]:
pred_log = model.predict(X_test_all)
pred_sales = np.expm1(pred_log)

pred_sales = np.where(
    model_test_without_sales["Open"] == 0,
    0,
    pred_sales
)

xgb_score = rmspe(y_test_true, pred_sales)

print("Baseline RMSPE:", baseline_score)
print("XGBoost RMSPE:", xgb_score)

Baseline RMSPE: 0.3071347823551685
XGBoost RMSPE: 0.1540919559443722


In [18]:
results = model_test.copy()
results["PredictedSales"] = pred_sales
results["AbsoluteError"] = abs(results["Sales"] - results["PredictedSales"])
results["PercentageError"] = results["AbsoluteError"] / results["Sales"].replace(0, np.nan)

results[["Store", "Date", "Sales", "PredictedSales", "AbsoluteError", "PercentageError"]].head(20)

,Store,Date,Sales,PredictedSales,AbsoluteError,PercentageError
0,1,2015-07-31,5263,5819.324219,556.324219,0.105705
1,2,2015-07-31,6064,6607.544434,543.544434,0.089635
2,3,2015-07-31,8314,9597.471680,1283.471680,0.154375
3,4,2015-07-31,13995,11819.055664,2175.944336,0.155480
4,5,2015-07-31,4822,6456.705566,1634.705566,0.339010
5,6,2015-07-31,5651,6300.577148,649.577148,0.114949
6,7,2015-07-31,15344,11771.368164,3572.631836,0.232836
7,8,2015-07-31,8492,7463.625977,1028.374023,0.121099
8,9,2015-07-31,8565,9172.848633,607.848633,0.070969
9,10,2015-07-31,7185,6886.433594,298.566406,0.041554


In [19]:
# ============================================================
# Add recent-history features and compare with previous model
# ============================================================

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    return np.sqrt(
        np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2)
    )


def add_date_features(df):
    df = df.copy()

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)
    df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)

    return df


def clean_missing_values(df):
    df = df.copy()

    df["CompetitionDistance"] = df["CompetitionDistance"].fillna(999999)

    for col in [
        "CompetitionOpenSinceMonth",
        "CompetitionOpenSinceYear",
        "Promo2SinceWeek",
        "Promo2SinceYear",
        "PromoInterval",
    ]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df["Open"] = df["Open"].fillna(1)

    return df


def add_competition_features(df):
    df = df.copy()

    df["CompetitionOpen"] = 12 * (
        df["Year"] - df["CompetitionOpenSinceYear"]
    ) + (
        df["Month"] - df["CompetitionOpenSinceMonth"]
    )

    df.loc[
        (df["CompetitionOpenSinceYear"] == 0)
        | (df["CompetitionOpenSinceMonth"] == 0),
        "CompetitionOpen"
    ] = 0

    df["CompetitionOpen"] = df["CompetitionOpen"].clip(lower=0)

    return df


def add_promo2_features(df):
    df = df.copy()

    df["Promo2Open"] = 12 * (
        df["Year"] - df["Promo2SinceYear"]
    ) + (
        df["WeekOfYear"] - df["Promo2SinceWeek"]
    ) / 4.0

    df.loc[
        (df["Promo2"] == 0)
        | (df["Promo2SinceYear"] == 0)
        | (df["Promo2SinceWeek"] == 0),
        "Promo2Open"
    ] = 0

    df["Promo2Open"] = df["Promo2Open"].clip(lower=0)

    return df


def add_recent_history_features(full_data, cutoff_date):
    """
    Adds leakage-safe recent-history features.

    Important:
    For a row at date t, rolling features use only dates before t.
    This avoids using the target value from the same day.
    """

    df = full_data.copy()
    df = df.sort_values(["Store", "Date"])

    # Use NaN for closed days / zero-sales days in historical averages.
    # Closed-store sales = 0 is not real demand.
    df["SalesForHistory"] = df["Sales"].where(
        (df["Open"] == 1) & (df["Sales"] > 0),
        np.nan
    )

    # Lag features by store
    df["SalesLag7"] = df.groupby("Store")["SalesForHistory"].shift(7)
    df["SalesLag14"] = df.groupby("Store")["SalesForHistory"].shift(14)
    df["SalesLag28"] = df.groupby("Store")["SalesForHistory"].shift(28)

    # Rolling means by store.
    # shift(1) ensures that today's Sales is not used to predict today's Sales.
    for window in [7, 14, 28, 56]:
        df[f"RollingMean{window}"] = (
            df.groupby("Store")["SalesForHistory"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=3).mean())
        )

    # Rolling median can be robust to outliers
    for window in [7, 28]:
        df[f"RollingMedian{window}"] = (
            df.groupby("Store")["SalesForHistory"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=3).median())
        )

    # Same weekday last week and two weeks ago
    # In daily data, shift(7) and shift(14) are usually same weekday
    df["SameWeekdayLastWeekSales"] = df.groupby("Store")["SalesForHistory"].shift(7)
    df["SameWeekdayTwoWeeksAgoSales"] = df.groupby("Store")["SalesForHistory"].shift(14)

    # Average sales by store and day of week, computed from training period only
    train_only = df[df["Date"] < cutoff_date].copy()

    store_dow_mean = (
        train_only
        .groupby(["Store", "DayOfWeek"])["SalesForHistory"]
        .mean()
        .rename("StoreDayOfWeekMeanSales")
        .reset_index()
    )

    store_mean = (
        train_only
        .groupby("Store")["SalesForHistory"]
        .mean()
        .rename("StoreMeanSales")
        .reset_index()
    )

    store_promo_mean = (
        train_only
        .groupby(["Store", "Promo"])["SalesForHistory"]
        .mean()
        .rename("StorePromoMeanSales")
        .reset_index()
    )

    store_month_mean = (
        train_only
        .groupby(["Store", "Month"])["SalesForHistory"]
        .mean()
        .rename("StoreMonthMeanSales")
        .reset_index()
    )

    df = df.merge(store_dow_mean, on=["Store", "DayOfWeek"], how="left")
    df = df.merge(store_mean, on="Store", how="left")
    df = df.merge(store_promo_mean, on=["Store", "Promo"], how="left")
    df = df.merge(store_month_mean, on=["Store", "Month"], how="left")

    # Ratios: recent sales compared with usual store sales
    df["RollingMean7_vs_StoreMean"] = df["RollingMean7"] / df["StoreMeanSales"]
    df["RollingMean28_vs_StoreMean"] = df["RollingMean28"] / df["StoreMeanSales"]

    # Fill recent-history missing values using store-level means
    history_cols = [
        "SalesLag7",
        "SalesLag14",
        "SalesLag28",
        "RollingMean7",
        "RollingMean14",
        "RollingMean28",
        "RollingMean56",
        "RollingMedian7",
        "RollingMedian28",
        "SameWeekdayLastWeekSales",
        "SameWeekdayTwoWeeksAgoSales",
        "StoreDayOfWeekMeanSales",
        "StoreMeanSales",
        "StorePromoMeanSales",
        "StoreMonthMeanSales",
        "RollingMean7_vs_StoreMean",
        "RollingMean28_vs_StoreMean",
    ]

    for col in history_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

    # First fill using store mean, then global mean
    global_mean_sales = train_only["SalesForHistory"].mean()

    for col in history_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df["StoreMeanSales"])
            df[col] = df[col].fillna(global_mean_sales)

    df = df.drop(columns=["SalesForHistory"])

    return df


# ------------------------------------------------------------
# 2. Reload clean data and create final-month hidden test set
# ------------------------------------------------------------

train = pd.read_csv(
    "data/train.csv",
    parse_dates=["Date"],
    dtype={"StateHoliday": str},
    low_memory=False
)

store = pd.read_csv("data/store.csv")

data = train.merge(store, on="Store", how="left")

cutoff_date = pd.Timestamp("2015-07-01")

# ------------------------------------------------------------
# 3. Feature engineering on the full labelled training file
#    We still use the cutoff to prevent leakage in aggregate features.
# ------------------------------------------------------------

data_fe = add_date_features(data)
data_fe = clean_missing_values(data_fe)
data_fe = add_competition_features(data_fe)
data_fe = add_promo2_features(data_fe)
data_fe = add_recent_history_features(data_fe, cutoff_date=cutoff_date)

# Split after feature engineering
model_train_fe2 = data_fe[data_fe["Date"] < cutoff_date].copy()
model_test_fe2 = data_fe[data_fe["Date"] >= cutoff_date].copy()

y_test_true = model_test_fe2["Sales"].copy()

# ------------------------------------------------------------
# 4. Baseline again: store mean from training period
# ------------------------------------------------------------

store_average_sales = (
    model_train_fe2[(model_train_fe2["Open"] == 1) & (model_train_fe2["Sales"] > 0)]
    .groupby("Store")["Sales"]
    .mean()
)

global_average_sales = (
    model_train_fe2[(model_train_fe2["Open"] == 1) & (model_train_fe2["Sales"] > 0)]
    ["Sales"]
    .mean()
)

baseline_pred = model_test_fe2["Store"].map(store_average_sales)
baseline_pred = baseline_pred.fillna(global_average_sales)

baseline_pred = np.where(
    model_test_fe2["Open"] == 0,
    0,
    baseline_pred
)

baseline_score_2 = rmspe(y_test_true, baseline_pred)

# ------------------------------------------------------------
# 5. Prepare XGBoost data
# ------------------------------------------------------------

categorical_cols = [
    "StoreType",
    "Assortment",
    "StateHoliday",
    "PromoInterval"
]

combined = pd.concat(
    [
        model_train_fe2.drop(columns=["Sales", "Customers"], errors="ignore"),
        model_test_fe2.drop(columns=["Sales", "Customers"], errors="ignore")
    ],
    axis=0
)

combined_encoded = pd.get_dummies(
    combined,
    columns=categorical_cols,
    dummy_na=True
)

X_train_all = combined_encoded.iloc[:len(model_train_fe2)].copy()
X_test_all = combined_encoded.iloc[len(model_train_fe2):].copy()

# Remove raw Date
X_train_all = X_train_all.drop(columns=["Date"], errors="ignore")
X_test_all = X_test_all.drop(columns=["Date"], errors="ignore")

y_train_all = model_train_fe2["Sales"].copy()

# Train only on open stores with positive sales
open_train_mask = (
    (model_train_fe2["Open"] == 1)
    & (model_train_fe2["Sales"] > 0)
)

X_train_open = X_train_all.loc[open_train_mask].copy()
y_train_open_log = np.log1p(y_train_all.loc[open_train_mask])

# ------------------------------------------------------------
# 6. Train XGBoost with recent-history features
# ------------------------------------------------------------

model_history = XGBRegressor(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_history.fit(X_train_open, y_train_open_log)

# ------------------------------------------------------------
# 7. Predict final month and evaluate
# ------------------------------------------------------------

pred_log_history = model_history.predict(X_test_all)
pred_sales_history = np.expm1(pred_log_history)

pred_sales_history = np.where(
    model_test_fe2["Open"] == 0,
    0,
    pred_sales_history
)

xgb_history_score = rmspe(y_test_true, pred_sales_history)

# ------------------------------------------------------------
# 8. Optional correction factor for log-target/RMSPE mismatch
# ------------------------------------------------------------

def find_best_correction(y_true, pred_log, open_values):
    best_correction = 0
    best_score = float("inf")

    for correction in np.linspace(-0.12, 0.05, 171):
        pred = np.expm1(pred_log + correction)

        pred = np.where(
            open_values == 0,
            0,
            pred
        )

        score = rmspe(y_true, pred)

        if score < best_score:
            best_score = score
            best_correction = correction

    return best_correction, best_score


best_corr, corrected_score = find_best_correction(
    y_true=y_test_true,
    pred_log=pred_log_history,
    open_values=model_test_fe2["Open"].values
)

pred_sales_history_corrected = np.expm1(pred_log_history + best_corr)

pred_sales_history_corrected = np.where(
    model_test_fe2["Open"] == 0,
    0,
    pred_sales_history_corrected
)

# ------------------------------------------------------------
# 9. Compare results
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "Baseline: store average",
        "XGBoost + recent-history features",
        "XGBoost + recent-history features + correction"
    ],
    "RMSPE": [
        baseline_score_2,
        xgb_history_score,
        corrected_score
    ]
})

print(comparison)
print()
print("Best correction factor:", best_corr)

# If previous variables exist from the earlier run, compare them too
try:
    previous_comparison = pd.DataFrame({
        "Model": [
            "Previous baseline",
            "Previous XGBoost without recent-history features",
            "New XGBoost with recent-history features",
            "New XGBoost with recent-history features + correction"
        ],
        "RMSPE": [
            baseline_score,
            xgb_score,
            xgb_history_score,
            corrected_score
        ]
    })

    print()
    print("Comparison with previous run:")
    print(previous_comparison)

except NameError:
    pass

# ------------------------------------------------------------
# 10. Store detailed prediction results for inspection
# ------------------------------------------------------------

results_history = model_test_fe2.copy()
results_history["PredictedSales"] = pred_sales_history_corrected
results_history["AbsoluteError"] = abs(
    results_history["Sales"] - results_history["PredictedSales"]
)
results_history["PercentageError"] = (
    results_history["AbsoluteError"]
    / results_history["Sales"].replace(0, np.nan)
)

print()
print("First few predictions:")
display(
    results_history[
        ["Store", "Date", "Sales", "PredictedSales", "AbsoluteError", "PercentageError", "Open", "Promo", "StateHoliday", "SchoolHoliday"]
    ].head(20)
)

print()
print("Worst percentage errors:")
display(
    results_history.sort_values("PercentageError", ascending=False)[
        ["Store", "Date", "Sales", "PredictedSales", "AbsoluteError", "PercentageError", "Open", "Promo", "StateHoliday", "SchoolHoliday"]
    ].head(20)
)

                                            Model     RMSPE
0                         Baseline: store average  0.307175
1               XGBoost + recent-history features  0.114399
2  XGBoost + recent-history features + correction  0.113455

Best correction factor: -0.015000000000000013

Comparison with previous run:
                                               Model     RMSPE
0                                  Previous baseline  0.307135
1   Previous XGBoost without recent-history features  0.154092
2           New XGBoost with recent-history features  0.114399
3  New XGBoost with recent-history features + cor...  0.113455

First few predictions:


,Store,Date,Sales,PredictedSales,AbsoluteError,PercentageError,Open,Promo,StateHoliday,SchoolHoliday
911,1,2015-07-01,5223,4976.558069,246.441931,0.047184,1,1,0,0
912,1,2015-07-02,5558,4978.295763,579.704237,0.104301,1,1,0,0
913,1,2015-07-03,4665,4668.218652,3.218652,0.000690,1,1,0,0
914,1,2015-07-04,4797,4439.408872,357.591128,0.074545,1,0,0,0
915,1,2015-07-05,0,0.000000,0.000000,NaN,0,0,0,0
916,1,2015-07-06,4359,3949.956085,409.043915,0.093839,1,0,0,0
917,1,2015-07-07,3650,3843.686433,193.686433,0.053065,1,0,0,0
918,1,2015-07-08,3797,3473.418386,323.581614,0.085220,1,0,0,0
919,1,2015-07-09,3897,3491.020869,405.979131,0.104177,1,0,0,0
920,1,2015-07-10,3808,3776.914345,31.085655,0.008163,1,0,0,0



Worst percentage errors:


,Store,Date,Sales,PredictedSales,AbsoluteError,PercentageError,Open,Promo,StateHoliday,SchoolHoliday
265106,292,2015-07-10,1012,3916.701700,2904.701700,2.870259,1,0,0,0
828279,909,2015-07-01,3547,13109.092324,9562.092324,2.695825,1,1,0,0
265100,292,2015-07-04,1646,5041.614273,3395.614273,2.062949,1,0,0,0
798863,876,2015-07-15,4739,10747.859875,6008.859875,1.267959,1,1,0,1
457031,501,2015-07-25,1571,3378.319027,1807.319027,1.150426,1,0,0,0
487174,534,2015-07-02,4488,8690.635335,4202.635335,0.936416,1,1,0,0
798861,876,2015-07-13,8088,14993.529476,6905.529476,0.853799,1,1,0,1
265099,292,2015-07-03,5036,9326.405071,4290.405071,0.851947,1,1,0,0
265103,292,2015-07-07,3077,5435.996404,2358.996404,0.766655,1,0,0,0
798852,876,2015-07-04,14708,3504.002946,11203.997054,0.761762,1,0,0,0
